# Green Cellpose traces from `Convert_seg_to_stat`

Loads the Suite2p-compatible outputs made by `Convert_seg_to_stat`, applies neuropil subtraction and a sliding-window baseline, then reproduces the trace and artifact-removal plots from `green_cellpose_extract_traces_and_plot_dff.ipynb`.

## Session 216, run 04 — MY_71_R
This is a separate copy of green_cellpose_suite2p_traces_and_plot_dff.ipynb. All input paths target session 216, run 04. Run cells in order in the mini2p-preprocessing environment. Saved output folders are specific to this workflow within that run. Existing output files in those folders are replaced when their save cells run. Trace-display ranges default to 2,000 frames where configurable; calculations use the whole recording.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import percentile_filter, binary_dilation

PLANE_DIR = Path('/Volumes/behrens/meg/3x3_field_magnitude_bandit/derivatives/cohort-01/sub-03_id-MY_71_R/ses-216_date-20260730/funcimg/run-04/suite2p/plane0')

# 'functional' loads F/Fneu; 'chan2' loads F_chan2/Fneu_chan2.
TRACE_SOURCE = 'functional'
OUTPUT_DIR = PLANE_DIR / 'cellpose_suite2p_sliding_dff'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NEUROPIL_COEFFICIENT = 0.7
FRAME_RATE_HZ = 7.52
BASELINE_WINDOW_SECONDS = 30.0
BASELINE_PERCENTILE = 20.0

if TRACE_SOURCE == 'functional':
    f_path, fneu_path = PLANE_DIR / 'F.npy', PLANE_DIR / 'Fneu.npy'
elif TRACE_SOURCE == 'chan2':
    f_path, fneu_path = PLANE_DIR / 'F_chan2.npy', PLANE_DIR / 'Fneu_chan2.npy'
else:
    raise ValueError("TRACE_SOURCE must be 'functional' or 'chan2'")

for path in [f_path, fneu_path, PLANE_DIR / 'stat.npy', PLANE_DIR / 'iscell.npy']:
    if not path.exists():
        raise FileNotFoundError(path)

print('Plane:', PLANE_DIR)
print('Trace source:', TRACE_SOURCE)
print('Output:', OUTPUT_DIR)


## Inspect a specific registered movie frame

In [ ]:
# Zero-based frame number: 0 is the first registered frame.
RAW_FRAME_NUMBER = 3000
FRAME_DISPLAY_PERCENTILES = (1.0, 99.5)

ops = np.load(PLANE_DIR / 'ops.npy', allow_pickle=True).item()
Ly, Lx = int(ops['Ly']), int(ops['Lx'])
binary_path = PLANE_DIR / ('data.bin' if TRACE_SOURCE == 'functional' else 'data_chan2.bin')
if not binary_path.exists():
    raise FileNotFoundError(binary_path)
bytes_per_frame = np.dtype(np.int16).itemsize * Ly * Lx
n_binary_frames = binary_path.stat().st_size // bytes_per_frame
if not 0 <= RAW_FRAME_NUMBER < n_binary_frames:
    raise IndexError(
        f'RAW_FRAME_NUMBER={RAW_FRAME_NUMBER} is outside 0–{n_binary_frames - 1}'
    )
registered_movie = np.memmap(
    binary_path, dtype=np.int16, mode='r', shape=(n_binary_frames, Ly, Lx)
)
raw_frame = np.asarray(registered_movie[RAW_FRAME_NUMBER])
finite_pixels = raw_frame[np.isfinite(raw_frame)]
vmin, vmax = np.percentile(finite_pixels, FRAME_DISPLAY_PERCENTILES)

fig, ax = plt.subplots(figsize=(8, 8))
image = ax.imshow(raw_frame, cmap='gray', vmin=vmin, vmax=vmax)
ax.set_title(
    f'{TRACE_SOURCE} registered frame {RAW_FRAME_NUMBER} '
    f'({FRAME_DISPLAY_PERCENTILES[0]:g}–{FRAME_DISPLAY_PERCENTILES[1]:g} percentile)'
)
ax.axis('off')
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label='Registered intensity')
fig.tight_layout()
plt.show()


### Save a registered-frame range as a TIFF movie

In [ ]:
import tifffile

MOVIE_START_FRAME = 2500
MOVIE_END_FRAME = 3500  # inclusive
MOVIE_FPS = FRAME_RATE_HZ
MOVIE_DISPLAY_PERCENTILES = (1.0, 99.5)
MOVIE_OUTPUT = OUTPUT_DIR / (
    f'{TRACE_SOURCE}_registered_frames_{MOVIE_START_FRAME}_{MOVIE_END_FRAME}.tiff'
)

movie_ops = np.load(PLANE_DIR / 'ops.npy', allow_pickle=True).item()
movie_Ly, movie_Lx = int(movie_ops['Ly']), int(movie_ops['Lx'])
movie_binary_path = PLANE_DIR / (
    'data.bin' if TRACE_SOURCE == 'functional' else 'data_chan2.bin'
)
movie_bytes_per_frame = np.dtype(np.int16).itemsize * movie_Ly * movie_Lx
movie_n_frames = movie_binary_path.stat().st_size // movie_bytes_per_frame
if not 0 <= MOVIE_START_FRAME <= MOVIE_END_FRAME < movie_n_frames:
    raise IndexError(
        f'Requested inclusive range {MOVIE_START_FRAME}–{MOVIE_END_FRAME}; '
        f'available frames are 0–{movie_n_frames - 1}'
    )
movie_data = np.memmap(
    movie_binary_path, dtype=np.int16, mode='r',
    shape=(movie_n_frames, movie_Ly, movie_Lx),
)

# Estimate one fixed contrast range from evenly spaced frames so brightness
# does not fluctuate because of per-frame autoscaling.
contrast_sample_indices = np.linspace(
    MOVIE_START_FRAME, MOVIE_END_FRAME,
    min(50, MOVIE_END_FRAME - MOVIE_START_FRAME + 1), dtype=int,
)
contrast_sample = np.asarray(movie_data[contrast_sample_indices])
movie_vmin, movie_vmax = np.percentile(
    contrast_sample, MOVIE_DISPLAY_PERCENTILES
)
if not movie_vmax > movie_vmin:
    raise ValueError('Movie contrast range is empty.')

with tifffile.TiffWriter(MOVIE_OUTPUT, bigtiff=True) as writer:
    for frame_index in range(MOVIE_START_FRAME, MOVIE_END_FRAME + 1):
        frame = np.asarray(movie_data[frame_index], dtype=np.float32)
        frame = np.clip((frame - movie_vmin) / (movie_vmax - movie_vmin), 0, 1)
        writer.write(
            np.round(frame * 255).astype(np.uint8),
            photometric='minisblack',
            metadata={'fps': MOVIE_FPS},
            contiguous=False,
        )

print(
    f'Saved {MOVIE_END_FRAME - MOVIE_START_FRAME + 1} frames at '
    f'{MOVIE_FPS:g} fps to {MOVIE_OUTPUT}'
)


## Load converted Suite2p traces and calculate ΔF/F

In [ ]:
F_cell = np.load(f_path).astype(np.float32)
F_neuropil = np.load(fneu_path).astype(np.float32)
stat = np.load(PLANE_DIR / 'stat.npy', allow_pickle=True)
iscell = np.load(PLANE_DIR / 'iscell.npy')

if F_cell.shape != F_neuropil.shape:
    raise ValueError(f'F shape {F_cell.shape} != Fneu shape {F_neuropil.shape}')
if F_cell.shape[0] != len(stat):
    raise ValueError(f'{F_cell.shape[0]} traces but {len(stat)} stat entries')

F_corrected = (F_cell - NEUROPIL_COEFFICIENT * F_neuropil).astype(np.float32)
window_frames = int(round(FRAME_RATE_HZ * BASELINE_WINDOW_SECONDS))
if window_frames < 1:
    raise ValueError('Baseline window must contain at least one frame')

F0_sliding = percentile_filter(
    F_corrected,
    percentile=BASELINE_PERCENTILE,
    size=(1, window_frames),
    mode='nearest',
)
valid_baseline = np.isfinite(F0_sliding) & (F0_sliding > 1e-6)
dff = np.full(F_corrected.shape, np.nan, dtype=np.float32)
np.divide(F_corrected - F0_sliding, F0_sliding, out=dff, where=valid_baseline)
valid_roi_indices = np.flatnonzero(np.any(np.isfinite(dff), axis=1))
invalid_roi_indices = np.setdiff1d(np.arange(dff.shape[0]), valid_roi_indices)

print('ROIs × frames:', dff.shape)
print(f'Baseline window: {window_frames} frames ({BASELINE_WINDOW_SECONDS:g} s)')
print(f'Valid ROIs: {len(valid_roi_indices)} / {dff.shape[0]}')
print('Excluded all-NaN ROI IDs:', invalid_roi_indices.tolist())
print('Finite ΔF/F range:', np.nanmin(dff), np.nanmax(dff))


In [ ]:
np.save(OUTPUT_DIR / 'F_cell.npy', F_cell)
np.save(OUTPUT_DIR / 'F_neuropil.npy', F_neuropil)
np.save(OUTPUT_DIR / 'F.npy', F_corrected)
np.save(OUTPUT_DIR / 'F0.npy', F0_sliding)
np.save(OUTPUT_DIR / 'dff.npy', dff)
np.save(OUTPUT_DIR / 'stat.npy', stat)
np.save(OUTPUT_DIR / 'iscell.npy', iscell)
np.save(OUTPUT_DIR / 'valid_dff_rois.npy', np.any(np.isfinite(dff), axis=1))
print('Saved neuropil-corrected sliding-window outputs to:', OUTPUT_DIR)


## Individual and percentile-scaled traces

In [ ]:
ROI_TO_PLOT = int(valid_roi_indices[0])
plt.figure(figsize=(14, 4))
plt.plot(dff[ROI_TO_PLOT], lw=0.8)
plt.xlabel('Frame')
plt.ylabel('ΔF/F')
plt.title(f'ROI {ROI_TO_PLOT}: neuropil-corrected sliding-window ΔF/F')
plt.tight_layout()
plt.show()


In [ ]:
N_ROIS_TO_PLOT = 30
START_FRAME = 0
END_FRAME = 2000  # use None for the full recording
LOW_PCTL = 1
HIGH_PCTL = 95
rois_to_plot = valid_roi_indices[:N_ROIS_TO_PLOT]
fig, axes = plt.subplots(len(rois_to_plot), 1, figsize=(15, 1.5 * len(rois_to_plot)), sharex=True)
axes = np.atleast_1d(axes)
for ax, roi in zip(axes, rois_to_plot):
    trace = dff[roi]
    ax.plot(trace, lw=0.7)
    ymin, ymax = np.nanpercentile(trace, [LOW_PCTL, HIGH_PCTL])
    pad = 0.1 * (ymax - ymin) if ymax > ymin else 1.0
    ax.set_ylim(ymin - pad, ymax + pad)
    ax.set_ylabel(str(roi), rotation=0, labelpad=20)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlim(START_FRAME, END_FRAME)
axes[-1].set_xlabel('Frame')
fig.suptitle('ΔF/F traces, 1st–95th percentile-scaled')
fig.tight_layout()
plt.show()


## Stacked traces in original ROI order

In [ ]:
def plot_stacked_traces(traces, roi_indices, start_frame=0, end_frame=2000, low=1, high=95, spacing=1.5, title='Stacked traces'):
    roi_indices = np.asarray(roi_indices, dtype=int)
    frame_slice = slice(start_frame, end_frame)
    frames = np.arange(traces.shape[1])[frame_slice]
    offsets = np.arange(len(roi_indices)) * spacing
    colors = plt.cm.tab10(np.arange(len(roi_indices)) % 10)
    fig, ax = plt.subplots(figsize=(16, max(6, 0.45 * len(roi_indices))))
    for offset, color, roi in zip(offsets, colors, roi_indices):
        trace = np.asarray(traces[roi, frame_slice], dtype=float)
        lo, hi = np.nanpercentile(trace, [low, high])
        scale = hi - lo
        if not np.isfinite(scale) or scale <= 0:
            continue
        shown = np.clip((trace - lo) / scale, -0.25, 1.25)
        ax.plot(frames, shown + offset, color=color, lw=0.65)
    ax.set_yticks(offsets + 0.5, [f'ROI {roi}' for roi in roi_indices])
    ax.set(xlabel='Imaging frame', ylabel='ROI', title=title)
    ax.spines[['top', 'right']].set_visible(False)
    ax.margins(x=0.01, y=0.03)
    fig.tight_layout()
    plt.show()

N_STACKED_ROIS = 20
plot_stacked_traces(
    dff, valid_roi_indices[:N_STACKED_ROIS],
    title=f'First {min(N_STACKED_ROIS, len(valid_roi_indices))} valid ΔF/F traces'
)


## Detect and interpolate population-wide artifacts

In [ ]:
ARTIFACT_DETECTION_METHOD = 'derivative_z'  # or 'percentile'
CELL_SPIKE_THRESHOLD = 6.0
LOW_ARTIFACT_PERCENTILE = 1.0
HIGH_ARTIFACT_PERCENTILE = 99.0
MIN_FRACTION_CELLS = 0.25
ARTIFACT_PADDING_FRAMES = 1

def detect_global_spike_frames(traces, method='derivative_z', cell_spike_threshold=6, low_percentile=1, high_percentile=99, min_fraction_cells=0.25, padding_frames=1):
    traces = np.asarray(traces, dtype=np.float64)
    valid_cells = np.any(np.isfinite(traces), axis=1)
    derivative = np.diff(traces, axis=1, prepend=traces[:, :1])
    med = np.nanmedian(derivative, axis=1, keepdims=True)
    mad = np.nanmedian(np.abs(derivative - med), axis=1, keepdims=True)
    scale = 1.4826 * mad
    scale[scale < 1e-12] = np.nan
    derivative_z = (derivative - med) / scale
    if method == 'derivative_z':
        cells_spiking = np.abs(derivative_z) >= cell_spike_threshold
    elif method == 'percentile':
        lo = np.nanpercentile(traces, low_percentile, axis=1, keepdims=True)
        hi = np.nanpercentile(traces, high_percentile, axis=1, keepdims=True)
        cells_spiking = (traces < lo) | (traces > hi)
    else:
        raise ValueError("method must be 'derivative_z' or 'percentile'")
    fraction_spiking = np.mean(cells_spiking[valid_cells], axis=0)
    artifact_frames = fraction_spiking >= min_fraction_cells
    if padding_frames > 0:
        artifact_frames = binary_dilation(artifact_frames, structure=np.ones(2 * padding_frames + 1, bool))
    return artifact_frames, fraction_spiking, derivative_z

def interpolate_artifact_frames(traces, artifact_frames):
    traces = np.asarray(traces, dtype=np.float64)
    corrected = traces.copy()
    frames = np.arange(traces.shape[1])
    for roi in range(traces.shape[0]):
        valid = ~artifact_frames & np.isfinite(traces[roi])
        if valid.sum() >= 2:
            corrected[roi, artifact_frames] = np.interp(frames[artifact_frames], frames[valid], traces[roi, valid])
    return corrected.astype(np.float32)

artifact_frames, fraction_spiking, derivative_z = detect_global_spike_frames(
    dff, method=ARTIFACT_DETECTION_METHOD, cell_spike_threshold=CELL_SPIKE_THRESHOLD,
    low_percentile=LOW_ARTIFACT_PERCENTILE, high_percentile=HIGH_ARTIFACT_PERCENTILE,
    min_fraction_cells=MIN_FRACTION_CELLS, padding_frames=ARTIFACT_PADDING_FRAMES,
)
dff_corrected = interpolate_artifact_frames(dff, artifact_frames)
np.save(OUTPUT_DIR / 'dff_corrected.npy', dff_corrected)
np.save(OUTPUT_DIR / 'artifact_frames.npy', artifact_frames)
print(f'Detected {artifact_frames.sum()} / {len(artifact_frames)} artifact frames ({100 * artifact_frames.mean():.3f}%)')


### View registered frames identified as artifacts

In [ ]:
MAX_ARTIFACT_FRAMES_TO_PLOT = 8
ARTIFACT_FRAME_DISPLAY_PERCENTILES = (1.0, 99.5)
CONTEXT_FRAMES_EACH_SIDE = 100

# Reopen the selected registered binary so this cell depends only on the
# path settings and the artifact mask, not the earlier preview cell.
ops_for_artifacts = np.load(PLANE_DIR / 'ops.npy', allow_pickle=True).item()
artifact_Ly, artifact_Lx = int(ops_for_artifacts['Ly']), int(ops_for_artifacts['Lx'])
artifact_binary_path = PLANE_DIR / (
    'data.bin' if TRACE_SOURCE == 'functional' else 'data_chan2.bin'
)
artifact_bytes_per_frame = np.dtype(np.int16).itemsize * artifact_Ly * artifact_Lx
artifact_binary_n_frames = artifact_binary_path.stat().st_size // artifact_bytes_per_frame
artifact_movie = np.memmap(
    artifact_binary_path, dtype=np.int16, mode='r',
    shape=(artifact_binary_n_frames, artifact_Ly, artifact_Lx),
)
artifact_indices = np.flatnonzero(artifact_frames)
artifact_indices = artifact_indices[artifact_indices < artifact_binary_n_frames]

if len(artifact_indices) == 0:
    print('No artifact frames were detected.')
else:
    if len(artifact_indices) > MAX_ARTIFACT_FRAMES_TO_PLOT:
        selection = np.linspace(
            0, len(artifact_indices) - 1, MAX_ARTIFACT_FRAMES_TO_PLOT
        ).round().astype(int)
        shown_artifact_indices = artifact_indices[selection]
    else:
        shown_artifact_indices = artifact_indices

    shown_frames = np.asarray(artifact_movie[shown_artifact_indices])
    context_means = []
    context_counts = []
    for frame_index in shown_artifact_indices:
        before = np.arange(
            max(0, frame_index - CONTEXT_FRAMES_EACH_SIDE), frame_index
        )
        after = np.arange(
            frame_index + 1,
            min(artifact_binary_n_frames, frame_index + CONTEXT_FRAMES_EACH_SIDE + 1),
        )
        context_indices = np.concatenate([before, after])
        context_means.append(
            np.asarray(artifact_movie[context_indices], dtype=np.float32).mean(axis=0)
        )
        context_counts.append(len(context_indices))
    context_means = np.stack(context_means)

    fig, axes = plt.subplots(
        len(shown_artifact_indices), 2,
        figsize=(8, 3.5 * len(shown_artifact_indices)), squeeze=False,
    )
    for row, (frame_index, frame, context_mean, context_count) in enumerate(zip(
        shown_artifact_indices, shown_frames, context_means, context_counts
    )):
        # Scale each image robustly so structure is visible in both the
        # instantaneous frame and the lower-noise contextual mean.
        raw_limits = np.percentile(frame, ARTIFACT_FRAME_DISPLAY_PERCENTILES)
        mean_limits = np.percentile(context_mean, ARTIFACT_FRAME_DISPLAY_PERCENTILES)
        axes[row, 0].imshow(
            frame, cmap='gray', vmin=raw_limits[0], vmax=raw_limits[1]
        )
        axes[row, 1].imshow(
            context_mean, cmap='gray', vmin=mean_limits[0], vmax=mean_limits[1]
        )
        axes[row, 0].set_title(f'Artifact frame {frame_index}')
        axes[row, 1].set_title(
            f'Mean of {context_count} surrounding frames '
        )
        axes[row, 0].axis('off')
        axes[row, 1].axis('off')
    fig.suptitle(
        f'Artifact frames and surrounding-frame means '
        f'({len(shown_artifact_indices)} of {len(artifact_indices)} shown)', y=1.01
    )
    fig.tight_layout()
    plt.show()


In [ ]:
x = np.arange(dff.shape[1]) / FRAME_RATE_HZ
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(x, fraction_spiking, lw=0.8)
ax.axhline(MIN_FRACTION_CELLS, color='crimson', ls='--', label='Artifact threshold')
idx = np.flatnonzero(artifact_frames)
if len(idx):
    ax.scatter(x[idx], fraction_spiking[idx], s=8, label='Removed frames')
ax.set(xlabel='Time (s)', ylabel='Fraction of cells', title='Population-wide artifact detection')
ax.spines[['top', 'right']].set_visible(False)
ax.legend()
fig.tight_layout()
plt.show()


## Corrected traces in original ROI order (no responsiveness ranking)

In [ ]:
corrected_valid_rois = np.flatnonzero(np.any(np.isfinite(dff_corrected), axis=1))
plot_stacked_traces(
    dff_corrected, corrected_valid_rois[:N_STACKED_ROIS],
    title=f'First {min(N_STACKED_ROIS, len(corrected_valid_rois))} valid traces after artifact removal'
)


## Optional: most responsive corrected cells

In [ ]:
N_RESPONSIVE_ROIS = 20
RESPONSE_PERCENTILE = 99
response_score = np.full(dff_corrected.shape[0], np.nan)
response_score[corrected_valid_rois] = np.nanpercentile(dff_corrected[corrected_valid_rois], RESPONSE_PERCENTILE, axis=1)
top_rois = corrected_valid_rois[np.argsort(response_score[corrected_valid_rois])[::-1]][:N_RESPONSIVE_ROIS]
fig, axes = plt.subplots(len(top_rois), 1, figsize=(15, 1.5 * len(top_rois)), sharex=True)
axes = np.atleast_1d(axes)
for ax, roi in zip(axes, top_rois):
    trace = dff_corrected[roi]
    ax.plot(trace, lw=0.7)
    lo, hi = np.nanpercentile(trace, [LOW_PCTL, HIGH_PCTL])
    pad = 0.1 * (hi - lo) if hi > lo else 1.0
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_ylabel(f'ROI {roi}', rotation=0, labelpad=28)
    ax.spines[['top', 'right']].set_visible(False)
axes[-1].set_xlabel('Frame')
fig.suptitle(f'Top {len(top_rois)} cells by corrected {RESPONSE_PERCENTILE}th-percentile ΔF/F')
fig.tight_layout()
plt.show()
